In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# NodePort of the `spark-connect` Service. Inside the cluster: sc://spark-connect:15002
spark = SparkSession.builder.remote("sc://192.168.49.2:31502").getOrCreate()
spark.version

'4.1.4'

In [17]:
spark.sql("SHOW schemas IN lake").show()

+---------+
|namespace|
+---------+
|   bronze|
+---------+



In [21]:
spark.sql("create schema if not exists lake.bronze").show()
spark.sql("SHOW schemas from lake").show()

++
||
++
++

+---------+
|namespace|
+---------+
|   bronze|
+---------+



In [22]:
spark.sql("""create table if not exists lake.bronze.users
            (
                id integer,
                created_at timestamp,
                name string                
            )
            using iceberg
            PARTITIONED BY (day(created_at))
          """).show()
spark.sql("SHOW tables from lake.bronze").show()

++
||
++
++

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   bronze|    users|      false|
+---------+---------+-----------+



## What's in the catalog

In [23]:
spark.sql("select * from lake.bronze.users").show()

+---+----------+----+
| id|created_at|name|
+---+----------+----+
+---+----------+----+



In [26]:
spark.sql("""
      INSERT INTO lake.bronze.users (id, created_at, name)
      VALUES (1, TIMESTAMP '2026-11-23 03:06:48', 'reza')
  """).show()

++
||
++
++



In [28]:
spark.sql("select * from lake.bronze.users").show()

+---+-------------------+----+
| id|         created_at|name|
+---+-------------------+----+
|  1|2026-11-23 03:06:48|reza|
+---+-------------------+----+



In [32]:
spark.sql("SELECT * from lake.bronze.users.files").show()

+-------+-----------------------------------------------------------------------------------------------------------------------+-----------+-------+------------+------------+------------------+---------------------------+------------------------+------------------------+----------------+------------------------------------------------------------------------+------------------------------------------------------------------------+------------+-------------+------------+-------------+------------+--------------------+--------------+---------------------+------------------------------------------------------------------------------------------------------------------+
|content|file_path                                                                                                              |file_format|spec_id|partition   |record_count|file_size_in_bytes|column_sizes               |value_counts            |null_value_counts       |nan_value_counts|lower_bounds                       

In [34]:
spark.sql("""
      INSERT INTO lake.bronze.users (id, created_at, name)
      VALUES (2, TIMESTAMP '2026-11-23 04:06:48', 'ali'),
              (3, TIMESTAMP '2026-11-23 04:06:48', 'mamad')
  """).show()

++
||
++
++



In [39]:
spark.sql("""
      INSERT INTO lake.bronze.users (id, created_at, name)
      VALUES (4, TIMESTAMP '2026-11-23 04:06:48', 'jafar')
  """).show()

++
||
++
++



In [41]:
spark.sql("""
      INSERT INTO lake.bronze.users (id, created_at, name)
      VALUES (5, TIMESTAMP '2026-11-23 04:06:48', 'akbar')
  """).show()

++
||
++
++



In [43]:
spark.sql("""
      INSERT INTO lake.bronze.users (id, created_at, name)
      VALUES (6, TIMESTAMP '2026-11-23 04:07:48', 'asghar')
  """).show()

++
||
++
++



In [44]:
spark.sql("select * from lake.bronze.users").show()

+---+-------------------+------+
| id|         created_at|  name|
+---+-------------------+------+
|  4|2026-11-23 04:06:48| jafar|
|  2|2026-11-23 04:06:48|   ali|
|  3|2026-11-23 04:06:48| mamad|
|  6|2026-11-23 04:07:48|asghar|
|  5|2026-11-23 04:06:48| akbar|
|  1|2026-11-23 03:06:48|  reza|
+---+-------------------+------+



In [45]:
spark.sql("SELECT * from lake.bronze.users.files").show()

+-------+--------------------+-----------+-------+------------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+------------+--------------------+--------------+---------------------+--------------------+
|content|           file_path|file_format|spec_id|   partition|record_count|file_size_in_bytes|        column_sizes|        value_counts|   null_value_counts|nan_value_counts|        lower_bounds|        upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|first_row_id|referenced_data_file|content_offset|content_size_in_bytes|    readable_metrics|
+-------+--------------------+-----------+-------+------------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+-

In [46]:
spark.sql("CALL lake.system.rewrite_data_files('bronze.users')").show()

+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|                         5|                     1|                 4663|                      0|                         0|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+



In [47]:
spark.sql("SELECT * from lake.bronze.users.files").show()

+-------+--------------------+-----------+-------+------------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+------------+--------------------+--------------+---------------------+--------------------+
|content|           file_path|file_format|spec_id|   partition|record_count|file_size_in_bytes|        column_sizes|        value_counts|   null_value_counts|nan_value_counts|        lower_bounds|        upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|first_row_id|referenced_data_file|content_offset|content_size_in_bytes|    readable_metrics|
+-------+--------------------+-----------+-------+------------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+-

In [48]:
spark.sql("""
      CALL lake.system.expire_snapshots(
          table       => 'bronze.users',
          older_than  => current_timestamp(),
          retain_last => 1
      )
  """).show()

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|                       5|                                  0|                                  0|                           5|                           5|                             0|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



In [50]:
spark.sql("""
      CALL lake.system.remove_orphan_files(
          table          => 'bronze.users',
          prefix_listing => true
      )
  """).show()

+--------------------+
|orphan_file_location|
+--------------------+
+--------------------+



In [51]:
spark.sql("select * from lake.bronze.users").show()

+---+-------------------+------+
| id|         created_at|  name|
+---+-------------------+------+
|  5|2026-11-23 04:06:48| akbar|
|  6|2026-11-23 04:07:48|asghar|
|  2|2026-11-23 04:06:48|   ali|
|  3|2026-11-23 04:06:48| mamad|
|  1|2026-11-23 03:06:48|  reza|
|  4|2026-11-23 04:06:48| jafar|
+---+-------------------+------+



In [55]:
daily = spark.table("lake.bronze.users")
daily.printSchema()

(daily
 .groupBy(F.to_date("created_at").alias("day"))
 .count()
 .orderBy("day")
 .show())

SparkConnectException: [NO_ACTIVE_SESSION] No active Spark session found. Please create a new Spark session before running the code.

In [54]:
spark.table("lake.bronze.users.partitions").show(truncate=False)

SparkConnectException: [NO_ACTIVE_SESSION] No active Spark session found. Please create a new Spark session before running the code.

Closes this client session only. The server keeps running for other notebooks.

In [ ]:
spark.stop()

# 